# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook demonstrates how to explore and process a Croissant-structured dataset—the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya—using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant metadata and dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n\n{metadata.description}\n")

## 2. Data Overview
Review available record sets, their fields, and unique `@id`s. 

All references in further code will use the actual `@id` values as required by best practices.

In [ ]:
# List record sets and their fields using @id
print("RecordSets present in this dataset:")

record_sets = []
for record_set in metadata.record_sets:
    print(f"  - RecordSet name: {record_set.name}")
    print(f"    @id: {record_set['@id']}")
    # List fields by @id and name
    print(f"    Fields:")
    for field in record_set.fields:
        print(f"      - {field.name}: @id={field['@id']}  (dataType: {field.data_type})")
    record_sets.append(record_set['@id'])
    print("")
# Store a reference to the first record set @id for later extraction
main_record_set_id = record_sets[0] if record_sets else None

## 3. Data Extraction
We now extract data for all available record sets, referencing each by its Croissant `@id`.

As an example, we will display the first record set and its fields. The data is loaded into pandas DataFrames for further analysis.

In [ ]:
# Extract all record sets into DataFrames using @id
dataframes = {}
for rset in record_sets:
    records = list(dataset.records(record_set=rset))
    # Only load DataFrame if there is data
    if records:
        dataframes[rset] = pd.DataFrame(records)

# For demonstration, print the columns of primary record set
if main_record_set_id in dataframes:
    print("Columns in main record set:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No data found for main record set. Check if record sets contain data.')

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field for analysis. We look for fields in the record set that are of type `Float` or `Integer` and then demonstrate filtering, normalization, and grouping. All variables and columns are referenced by their `@id` _as required_.

In [ ]:
# Identify numeric fields from the main record set for demonstration
numeric_field_id = None
group_field_id = None
numeric_candidates = []

for record_set in metadata.record_sets:
    if record_set['@id'] == main_record_set_id:
        for field in record_set.fields:
            if field.data_type in ['Float', 'Integer', 'Number']:
                numeric_candidates.append({'name': field.name, 'id': field['@id']})
        break

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]['id']
    print(f"Using field: '{numeric_candidates[0]['name']}' (@id = {numeric_field_id}) as numeric field.")
else:
    print('No numeric field found, EDA will be limited.')

# Try to select 'gender' or 'village' if present for grouping
group_candidates = ['gender', 'village', 'level_of_education']
df = dataframes[main_record_set_id] if main_record_set_id in dataframes else None
available_for_group = [c for c in group_candidates if (df is not None and c in df.columns)]
group_field_id = available_for_group[0] if available_for_group else None

if df is not None and numeric_field_id in df.columns:
    # Drop missing or non-numeric values for the demonstration
    threshold = df[numeric_field_id].quantile(0.90) # Use a high threshold as an example
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f} (90th percentile):\n")
    print(filtered_df.head())
    # Normalize the field
    colnorm = f"{numeric_field_id}_normalized"
    filtered_df[colnorm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized values for '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, colnorm]].head())
    # Grouping if group_field is present
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean of '{numeric_field_id}' grouped by '{group_field_id}':")
        print(grouped_df.head())
else:
    print(f"Main DataFrame is missing or numeric field '{numeric_field_id}' not found in columns.")

## 5. Visualization
Let's plot the numeric field distribution and, if available, the group-wise mean.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id:
        # Boxplot by group
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.show()
else:
    print('Insufficient data for visualization.')

## 6. Conclusion

This notebook demonstrated loading, overview, and basic analysis of the FAIR² Mental Health Survey dataset using `mlcroissant`.
By referencing all dataset entities by their Croissant `@id`s and dynamically accessing data fields, you ensure robust, reproducible workflows and precise processing. 

Further exploration can build on this by examining more complex multivariate relationships, handling missing data with advanced imputation, or connecting to healthcare data platforms for real-world use cases.